# Step 1 — Geometry

**Learning objective:** Understand the excavation domain and the 0.2 m impermeable wall cut-out.

**Run from top to bottom in VS Code.** Each notebook executes one workshop step and shows its output. Select the working OGS Postprocessing Python kernel.

**Prerequisites:** run earlier numbered steps. `GMSH2OGS` and `ogs` must be on PATH when used; Gmsh is imported from the selected Python kernel in Step 2. **Every image below represents a distinct quantity.** Green PASS messages verify the stated condition only.

In [ ]:
from pathlib import Path
import sys, subprocess
import numpy as np
import matplotlib.pyplot as plt
import meshio
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents, Path.cwd().parent) if (p / "models" / "excavation.prj").exists()), None)
assert ROOT is not None, "Open the extracted project directory in VS Code before running this notebook"
print('Training root:', ROOT)
print('Python:', sys.executable)
def run(*command):
    command = [str(c) for c in command]
    print('RUN:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=ROOT, check=True)


**Explain:** 65 × 30 m 2D domain. Wall centred at x = 32.5 m, thickness 0.2 m, toe y = 8 m. Open `geometry/excavation.geo` to inspect the Gmsh source. **Run:** visualize the geometry; this is a preview of the actual `.geo` file geometry, not an OGS result.

In [ ]:
import re
geo=(ROOT/'geometry'/'excavation.geo').read_text()
def parameter(name):
    match=re.search(r'^'+name+r'\s*=\s*([0-9.]+)\s*;',geo,re.M)
    assert match, f'Missing {name} in geometry file'
    return float(match.group(1))
L,H,c,w,toe=[parameter(p) for p in ('L','H','wall_x','wall_width','wall_toe')]
from matplotlib.patches import Polygon, Rectangle
verts=[(0,0),(L,0),(L,H),(c+w/2,H),(c+w/2,toe),(c-w/2,toe),(c-w/2,H),(0,H)]
fig,ax=plt.subplots(figsize=(12,5.5))
ax.add_patch(Polygon(verts,closed=True,facecolor='#dbeafe',edgecolor='#1e3a8a',linewidth=2))
ax.add_patch(Rectangle((c-w/2,toe),w,H-toe,color='white',ec='black',lw=1.5))
ax.annotate('Wall tip',xy=(c,toe),xytext=(c+5,toe+5),arrowprops={'arrowstyle':'->'})
ax.annotate('Top left BC: u=60',xy=(c/2,H),xytext=(5,H+3),arrowprops={'arrowstyle':'->'})
ax.annotate('Top right BC: u=30',xy=((c+L)/2,H),xytext=(44,H+3),arrowprops={'arrowstyle':'->'})
ax.set(xlim=(-2,L+2),ylim=(-1,H+7),xlabel='x [m]',ylabel='y [m]',title='Gmsh domain geometry (source preview)')
ax.set_aspect('equal');ax.grid(alpha=.2);plt.show()
print(f'Geometry: {L:g} × {H:g} m; wall thickness={w:g} m; toe={toe:g} m')

**Interpretation / PASS:** The model is a 65 × 30 m two-dimensional material domain with a 0.2 m wide opening from the upper boundary down to y = 8 m. The white slit is **excluded material**, not an extra soil subdomain. This drawing is geometry only: no mesh and no computed potential. **Next:** generate a triangular finite-element mesh.